# Analyzing N-Gram Distribution for Given Sequence for Procedural Learning

## Imports

In [ ]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import time
import json
from scipy.optimize import minimize, LinearConstraint
from scipy.stats import entropy as scipy_entropy
from itertools import combinations
from fractions import Fraction
from collections import Counter, defaultdict
from pathlib import Path

## Utility Functions

In [ ]:
# ENTROPY AND TRANSITION MATRIX
def entropy(probs):
    """Calculate Shannon entropy in bits"""
    probs = np.asarray(probs)
    p = probs[probs > 0]
    return -np.sum(p * np.log2(p))


def get_stationary_distribution(P):
    """Compute stationary distribution of Markov chain"""
    eigenvalues, eigenvectors = np.linalg.eig(P.T)
    idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-10)
    pi = np.real(eigenvectors[:, idx])
    pi = pi / pi.sum()
    return pi


def is_doubly_stochastic(P, tol=1e-6):
    """Check if matrix is doubly stochastic"""
    row_sums = P.sum(axis=1)
    col_sums = P.sum(axis=0)
    return (np.allclose(row_sums, 1.0, atol=tol) and
            np.allclose(col_sums, 1.0, atol=tol))


def display_matrix(P, name="Transition Matrix"):
    """Display transition matrix with computed properties"""
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    df = pd.DataFrame(P,
                     index=[f"From {i+1}" for i in range(len(P))],
                     columns=[f"To {j+1}" for j in range(len(P))])
    print(df)

    # Properties
    pi = get_stationary_distribution(P)
    H = [entropy(row) for row in P]

    print(f"\nStationary distribution π: {pi}")
    print(f"Uniform? {np.allclose(pi, 1.0/len(P))}")

    print(f"\nRow sums: {P.sum(axis=1)}")
    print(f"Col sums: {P.sum(axis=0)}")
    print(f"Doubly stochastic? {is_doubly_stochastic(P)}")

    print(f"\nConditional entropies:")
    for i, h in enumerate(H):
        print(f"  Position {i+1}: {h:.3f} bits")

    return pi, H

def calc_entropy(P, pos):
    probs = P[pos][P[pos] > 1e-10]
    return scipy_entropy(probs, base=2)

def sort_by_entropy(P, output_path=None):
    """Sort matrix rows/columns by ascending entropy."""
    n = len(P)

    # Calculate entropies
    entropies = [calc_entropy(P, i) for i in range(n)]
    print(f"Original entropies: {[f'{h:.3f}' for h in entropies]}")

    # Get permutation
    perm = np.argsort(entropies)

    # Apply to both rows and columns
    P_sorted = P[perm, :][:, perm]

    # Verify
    entropies_sorted = [calc_entropy(P_sorted, i) for i in range(n)]
    print(f"Sorted entropies: {[f'{h:.3f}' for h in entropies_sorted]}")
    print(f"Doubly stochastic: rows={np.allclose(P_sorted.sum(axis=1), 1.0)}, "
          f"cols={np.allclose(P_sorted.sum(axis=0), 1.0)}")

    if output_path:
        np.save(output_path, P_sorted)
        print(f"Saved to {output_path}")

    return P_sorted

In [ ]:
# GENERATE SEQUENCES
def generate_asrt_sequence(pattern, n_trials=600):
  """Generate traditional ASRT sequence (P-r-P-r alternating)"""
  sequence = []
  positions = [1, 2, 3, 4]

  for i in range(n_trials):
      if i % 2 == 0:  # Pattern trial
          sequence.append(pattern[i // 2 % len(pattern)])
      else:  # Random trial
          sequence.append(int((np.random.choice(positions))))

  return sequence


def generate_novel_sequence(matrix, n_trials=600):
    """Generate novel graded sequence based on transition matrix"""
    matrix_arr = np.array(matrix)

    # Validation checks
    if len(matrix_arr.shape) != 2:
        print("Transitional matrix is not 2-dimensional.")
        return
    if matrix_arr.shape[0] != matrix_arr.shape[1]:
        print("Transitional matrix is not square.")
        return

    # Force exact doubly stochastic by renormalizing
    # This ensures numerical precision issues don't cause problems
    matrix_arr = matrix_arr / matrix_arr.sum(axis=1, keepdims=True)  # Normalize rows

    positions = np.arange(matrix_arr.shape[0])
    sequence = np.zeros(n_trials, dtype=int)

    # Start from random position
    sequence[0] = np.random.choice(positions)

    # Generate sequence using transition probabilities
    for i in range(1, n_trials):
        current_pos = sequence[i-1]
        transition_probs = matrix_arr[current_pos].copy()  # Get row and copy

        # Force exact normalization for numerical stability
        transition_probs = transition_probs / transition_probs.sum()

        # Clip any negative values that might arise from floating point errors
        transition_probs = np.maximum(transition_probs, 0)
        transition_probs = transition_probs / transition_probs.sum()  # Renormalize after clipping

        sequence[i] = np.random.choice(positions, p=transition_probs)

    # Convert to 1-indexed (if needed for your experiment)
    return [x + 1 for x in sequence]

In [ ]:
# N-GRAM
def is_repetition(ngram, min_n=3):
    """Check if n-gram is a repetition (all elements the same)"""
    if len(ngram) < min_n:
      return False
    else:
      return len(set(ngram)) == 1


def is_trill(ngram):
    """Check if n-gram is a trill (A-B-A pattern)"""
    if len(ngram) != 3:
        return False
    return ngram[0] == ngram[2] and ngram[0] != ngram[1]


def filter_ngrams(ngrams, exclude_reps=True, exclude_trills=True):
    """
    Filter out repetitions and/or trills from n-grams

    Parameters:
    -----------
    ngrams : list of tuples
        List of n-grams
    exclude_reps : bool
        Whether to exclude repetitions (default: True)
    exclude_trills : bool
        Whether to exclude trills (default: True)

    Returns:
    --------
    filtered : list of tuples
        Filtered n-grams
    excluded_count : dict
        Counts of excluded types
    """

    filtered = []
    reps = 0
    rep_types = set()
    trills = 0
    trill_types = set()

    for ngram in ngrams:
        # Check for repetition
        if exclude_reps and is_repetition(ngram):
            reps += 1
            rep_types.add(ngram)
            continue

        # Check for trill (only for 3-grams)
        if exclude_trills and is_trill(ngram):
            trills += 1
            trill_types.add(ngram)
            continue

        # Keep it
        filtered.append(ngram)

    excluded_count = {
        'repetitions': reps,
        'repetition_types': rep_types,
        'repetition_type_count': len(rep_types),
        'trills': trills,
        'trill_types': trill_types,
        'trill_type_count': len(trill_types),
        'total_excluded': reps + trills,
        'original_count': len(ngrams),
        'filtered_count': len(filtered),
        'exclusion_rate': (reps + trills) / len(ngrams) * 100
    }

    return filtered, excluded_count


def get_ngrams(sequence, n):
    """Extract n-grams from sequence"""
    ngrams = []
    for i in range(len(sequence) - n + 1):
        ngrams.append(tuple(sequence[i:i+n]))
    return ngrams


def entropy_ratio(probs):
    K = len(probs)
    H = entropy(probs)
    H_max = np.log2(K)
    return H / H_max


def kl_from_uniform_via_entropy(probs):
  K = len(probs)
  return np.log2(K) - entropy(probs)


def kl_by_n(ngram_probs_by_n):
    """
    ngram_probs_by_n: dict {n: probs}
    """
    return {n: kl_from_uniform_via_entropy(probs)
            for n, probs in ngram_probs_by_n.items()}


def kl_growth(kl_by_n):
    ns = sorted(kl_by_n)
    return {n: kl_by_n[n] - kl_by_n[n-1]
            for n in ns[1:]}


def analyze_ngram_distribution_stats(sequence, n, exclude_reps=True, exclude_trills=True):
    """
    Get detailed statistics about n-gram probability distribution

    Parameters:
    -----------
    sequence : list
        Generated sequence
    n : int
        n-gram size to analyze
    exclude_reps : bool
        Whether to exclude repetitions
    exclude_trills : bool
        Whether to exclude trills

    Returns:
    --------
    stats : dict
        Dictionary with distribution statistics
    """

    # Get n-grams
    ngrams = get_ngrams(sequence, n)
    original_count = len(ngrams)

    # Filter if requested
    exclusion_info = None
    if exclude_reps or (exclude_trills and n == 3):
        ngrams, exclusion_info = filter_ngrams(ngrams, exclude_reps, exclude_trills)

    counts = Counter(ngrams)
    total = len(ngrams)

    if total == 0:
        print(f"Warning: No {n}-grams left after filtering!")
        return None

    probs = np.array([count / total for count in counts.values()])

    stats = {
        'n': n,
        'total_ngrams': total,
        'original_total': original_count,
        'unique_types': len(counts),
        'mean_prob': np.mean(probs),
        'median_prob': np.median(probs),
        'std_prob': np.std(probs),
        'min_prob': np.min(probs),
        'max_prob': np.max(probs),
        'range_ratio': np.max(probs) / np.min(probs),
        'entropy': -np.sum(probs * np.log2(probs + 1e-10)),
        'max_entropy': np.log2(len(counts)),
        'entropy_ratio': entropy_ratio(probs),
        'kl': kl_from_uniform_via_entropy(probs),
        'probabilities': probs,
        'ngram_counts': counts,
        'exclusion_info': exclusion_info
    }

    return stats


def print_distribution_summary(stats):
    """Print a nice summary of distribution statistics"""

    if stats is None:
        print("No stats available (all n-grams filtered out?)")
        return

    print(f"\n{'='*60}")
    print(f"{stats['n']}-GRAM DISTRIBUTION SUMMARY")
    print(f"{'='*60}")

    # Show exclusion info if available
    if stats['exclusion_info']:
        excl = stats['exclusion_info']
        print(f"\n🚫 Exclusions:")
        print(f"  Original count:         {excl['original_count']:,}")
        print(f"  Repetitions:            {excl['repetitions']:,}")
        print(f"  Reptitions Type Count:  {excl['repetition_type_count']:,}")
        print(f"  Trills:                 {excl['trills']:,}")
        print(f"  Trills Type Count:      {excl['trill_type_count']:,}")
        print(f"  Total excluded:         {excl['total_excluded']:,} ({excl['exclusion_rate']:.1f}%)")
        print(f"  Remaining:              {excl['filtered_count']:,}")

    print(f"\n📊 Basic Stats:")
    print(f"  Total {stats['n']}-grams:  {stats['total_ngrams']:,}")
    print(f"  Unique types:       {stats['unique_types']}")
    print(f"  Entropy:            {stats['entropy']:.3f} bits (max: {stats['max_entropy']:.3f})")
    print(f"  Entropy Ratio:      {stats['entropy_ratio']}")
    print(f"  KL Divergence:      {stats['kl']}")

    print(f"\n📈 Probability Distribution:")
    print(f"  Mean:               {stats['mean_prob']:.6f}")
    print(f"  Median:             {stats['median_prob']:.6f}")
    print(f"  Std dev:            {stats['std_prob']:.6f}")
    print(f"  Min:                {stats['min_prob']:.6f}")
    print(f"  Max:                {stats['max_prob']:.6f}")
    print(f"  Range (max/min):    {stats['range_ratio']:.1f}x")

    # Distribution shape analysis
    probs = stats['probabilities']
    q25 = np.percentile(probs, 25)
    q50 = np.percentile(probs, 50)
    q75 = np.percentile(probs, 75)

    very_high = np.sum(probs > q75)
    high = np.sum((probs > q50) & (probs <= q75))
    medium = np.sum((probs > q25) & (probs <= q50))
    low = np.sum(probs <= q25)

    print(f"\n🎯 Distribution Shape:")
    print(f"  Very high (>75%ile): {very_high} types")
    print(f"  High (50-75%ile):    {high} types")
    print(f"  Medium (25-50%ile):  {medium} types")
    print(f"  Low (<25%ile):       {low} types")


def get_ngram_distribution_summary_up_to_n(sequence, n, exclude_reps=True, exclude_trills=True):
  ngram_probs = {}
  for i in range(n):
    ngram_stats = analyze_ngram_distribution_stats(sequence, i+1, exclude_reps, exclude_trills)
    print_distribution_summary(ngram_stats)
    ngram_probs[i+1] = ngram_stats['probabilities']
  print(f"\n{'='*60}")
  print(f"\nKL Growth by n:")
  print(f"\n{'='*60}")
  print(kl_growth(kl_by_n(ngram_probs)))


In [ ]:
def analyze_markov_design(matrix=None, sequence=None, ngram_level=3, n_trials=600, exclude_reps=True, exclude_trills=True):
  if matrix: display_matrix(matrix)
  if matrix: get_stationary_distribution(matrix)
  seq = sequence if sequence else generate_novel_sequence(matrix, n_trials)
  seq_ngram_stats = analyze_ngram_distribution_stats(seq, ngram_level, exclude_reps, exclude_trills)
  seq_ngram_counts = seq_ngram_stats['ngram_counts']
  seq_ngram_probs = seq_ngram_stats['probabilities']
  seq_ngram_probs_3sf = map(lambda x: round(x, 2), seq_ngram_probs)
  seq_ngram_probs_3sf = list(seq_ngram_probs_3sf)
  seq_ngram_counts_probs = dict(zip(seq_ngram_counts.keys(), seq_ngram_probs_3sf))
  seq_ngram_probs_3sf.sort()
  seq_probs_triplet = defaultdict(list)
  for k, v in seq_ngram_counts_probs.items():
      seq_probs_triplet[float(v)].append(tuple(int(x) for x in k))
  seq_probs_triplet = dict(sorted(seq_probs_triplet.items()))
  get_ngram_distribution_summary_up_to_n(seq, 3)

  return Counter(seq_ngram_probs_3sf), seq_probs_triplet

## Design Criteria
(in decreasing order of importance)



```
1. UNIFORM UNIGRAM DISTRIBUTION [STRICT]
π₁ = π₂ = π₃ = π₄ = 1/4
πP = π
i.e. a doubly stochastic matrix (rows AND columns sum to 1)
This is important because we do not want to confound learning of transitional probabilities with learning positional frequency

2. INCREASING BRANCHING/GRADED ENTROPY ACROSS ROWS [CORE IDEA]
Strong formulation:
- Strictly increasing
- With first row being deterministic (only map to one state)
- And last row being fully undeterministic (can map to any state)

3. MINIMIZE DATA LOSS FROM REPETITIONS AND TRILLS
Repetitions (x-x-x) and trills (x-y-x) are thrown out in post-processing

4. UNIFORM (SUPPORTED) TRANSITION PROBABILITIES ON EACH ROW
All values in a row are equal to one another (unless zero)

```

# Traditional ASRT Design

In [ ]:
# Standard 4-element patterns (Howard and Howard, 1998; Janacsek, 2012) -- All possible permutations of [1,2,3,4] removing duplicates from wrapping
howard_asrt_pattern_1 = [1,2,3,4]
howard_asrt_pattern_2 = [1,2,4,3]
howard_asrt_pattern_3 = [1,3,2,4]
howard_asrt_pattern_4 = [1,3,4,2]
howard_asrt_pattern_5 = [1,4,2,3]
howard_asrt_pattern_6 = [1,4,3,2]

howard_asrt_seq_1 = generate_asrt_sequence(howard_asrt_pattern_1)
howard_asrt_seq_2 = generate_asrt_sequence(howard_asrt_pattern_2)
howard_asrt_seq_3 = generate_asrt_sequence(howard_asrt_pattern_3)
howard_asrt_seq_4 = generate_asrt_sequence(howard_asrt_pattern_4)
howard_asrt_seq_5 = generate_asrt_sequence(howard_asrt_pattern_5)
howard_asrt_seq_6 = generate_asrt_sequence(howard_asrt_pattern_6)

In [ ]:
howard_asrt_seq_1_triplet_stats = analyze_ngram_distribution_stats(howard_asrt_seq_1, 3, True, True)
howard_asrt_seq_1_triplet_counts = howard_asrt_seq_1_triplet_stats['ngram_counts']
howard_asrt_seq_1_triplet_probs = howard_asrt_seq_1_triplet_stats['probabilities']
howard_asrt_seq_1_triplet_probs_3sf = map(lambda x: round(x, 2), howard_asrt_seq_1_triplet_probs)
howard_asrt_seq_1_triplet_probs_3sf = list(howard_asrt_seq_1_triplet_probs_3sf)
howard_asrt_seq_1_triplet_counts_probs = dict(zip(howard_asrt_seq_1_triplet_counts.keys(), howard_asrt_seq_1_triplet_probs_3sf))
howard_asrt_seq_1_triplet_probs_3sf.sort()
howard_asrt_seq_1_probs_triplet = defaultdict(list)
for k, v in howard_asrt_seq_1_triplet_counts_probs.items():
    howard_asrt_seq_1_probs_triplet[float(v)].append(k)
howard_asrt_seq_1_probs_triplet = dict(sorted(howard_asrt_seq_1_probs_triplet.items()))

In [ ]:
Counter(howard_asrt_seq_1_triplet_probs_3sf)

Counter({np.float64(0.0): 6,
         np.float64(0.01): 22,
         np.float64(0.02): 3,
         np.float64(0.03): 2,
         np.float64(0.04): 6,
         np.float64(0.05): 6,
         np.float64(0.06): 2})

In [ ]:
get_ngram_distribution_summary_up_to_n(howard_asrt_seq_1, 5)


1-GRAM DISTRIBUTION SUMMARY

🚫 Exclusions:
  Original count:         600
  Repetitions:            0
  Reptitions Type Count:  0
  Trills:                 0
  Trills Type Count:      0
  Total excluded:         0 (0.0%)
  Remaining:              600

📊 Basic Stats:
  Total 1-grams:  600
  Unique types:       4
  Entropy:            1.996 bits (max: 2.000)
  Entropy Ratio:      0.9981959567693558
  KL Divergence:      0.0036080864612884067

📈 Probability Distribution:
  Mean:               0.250000
  Median:             0.259167
  Std dev:            0.017440
  Min:                0.220000
  Max:                0.261667
  Range (max/min):    1.2x

🎯 Distribution Shape:
  Very high (>75%ile): 0 types
  High (50-75%ile):    2 types
  Medium (25-50%ile):  1 types
  Low (<25%ile):       1 types

2-GRAM DISTRIBUTION SUMMARY

🚫 Exclusions:
  Original count:         599
  Repetitions:            0
  Reptitions Type Count:  0
  Trills:                 0
  Trills Type Count:      0
  Total excl

## **Summary Interpretation**

- Uniform unigram visitation: Min=24.8%, Max=25.1%
- Graded entropy at trigram level (excludes both repetitions and trills):
```
Counter({np.float64(0.01): 32, np.float64(0.04): 11, np.float64(0.05): 5})
# 16 high freq include both P-R-P and R-P-R
# 48 low freq include repetitions and trills
```
- Trigrams at each probability (excludes both repetitions and trills):
```
{0.01: [(1, 2, 3), (3, 3, 1), (1, 4, 3), (4, 4, 2),
  (2, 2, 4), (4, 3, 3), (4, 2, 2), (2, 3, 1), (1, 4, 4),
  (3, 3, 2), (2, 4, 1), (1, 1, 3), (3, 2, 2), (2, 3, 4),
  (1, 2, 4), (4, 3, 2), (2, 1, 4), (4, 2, 3), (2, 1, 1),
  (1, 3, 3), (3, 4, 1), (1, 1, 4), (2, 4, 4), (4, 1, 2),
  (3, 4, 2), (3, 1, 1), (4, 1, 3), (3, 2, 1), (1, 3, 4),
  (4, 4, 3), (3, 1, 2), (2, 2, 1)],
 0.04: [(1, 1, 2), (2, 3, 3), (3, 1, 4), (4, 3, 1), (2, 4, 3),
  (3, 4, 4), (4, 2, 1), (1, 2, 2), (3, 3, 4), (4, 1, 1),
  (2, 1, 3)],
 0.05: [(1, 3, 2), (3, 2, 4), (1, 4, 2), (2, 2, 3), (4, 4, 1)]}
```
- Entropy at trigram level: 5.144 / 5.585 bits
- Entropy ratio (unpredictability): 0.92
- Uniform supported transition probability values on each row: True ([1,0,0,0] for patterned trials; [1/2,1/2,1/2,1/2] for random trials)
- Data loss from repetitions & trills: 12.5%

# Josh's Graded Designs

## Core Design Idea

```
Four locations (1, 2, 3, 4), each with different conditional entropy:

Location 1: 0-CHOICE (DETERMINISTIC) (0 bits entropy)
  E.g. after 1: always goes to position 2 (100%)  

Location 2: 2-CHOICE (1 bit entropy)
  E.g. after 2: 2 → 3 (50%) or 2 → 4 (50%)

Location 3: 3-CHOICE (1.58 bits entropy)
  E.g. after 3: 3 → 1 (33%), 3 → 2 (33%), 3 → 4 (33%)

Location 4: 4-CHOICE (MAXIMALLY NONDETERMINISTIC) (2 bits entropy)
  E.g.: after 4: 4 → 1 (25%), 4 → 2 (25%), 4 → 3 (25%), 4 → 4 (25%)
```

## What we get to test

```
1. GRADED LEARNING
   • Can learners extract structure at different entropy levels?
   • Which conditions show fastest learning?
   • At what point does learning start to fail?
   -> What is the dynamic range of the procedural learning system?
   -> Is there a point where uncertainty becomes too high for the procedural system, and learning shifts to a different mechanism?

2. INDIVIDUAL DIFFERENCES
   • Do some people learn deterministic better?
   • Do others handle uncertainty better?
   -> What factors contribute to individual variability in procedural learning?

3. CRITICAL PERIOD EFFECTS
   • Is the developmental trajectory of procedural learning uniform across entropy evels?

4. GRAMMAR CORRELATION SPECIFICITY
   • Does grammar correlate equally with ALL entropy levels?
  ```

## Graded V1
```
P:
            TO 1      TO 2  TO 3      TO 4
FROM 1  0.000000  1.000000  0.00  0.000000
FROM 2  0.000000  0.000000  0.50  0.500000
FROM 3  0.333333  0.333333  0.00  0.333333
FROM 4  0.250000  0.250000  0.25  0.250000

π:
[0.15384615 0.30769231 0.23076923 0.30769231]
```



In [ ]:
graded_matrix_v1 = np.array([[0, 1, 0, 0],
          [0, 0, 1/2, 1/2],
          [1/3, 1/3, 0, 1/3],
          [1/4, 1/4, 1/4, 1/4]])
graded_matrix_v1_display = display_matrix(graded_matrix_v1)
graded_matrix_v1_p_pi = get_stationary_distribution(graded_matrix_v1)


Transition Matrix
            To 1      To 2  To 3      To 4
From 1  0.000000  1.000000  0.00  0.000000
From 2  0.000000  0.000000  0.50  0.500000
From 3  0.333333  0.333333  0.00  0.333333
From 4  0.250000  0.250000  0.25  0.250000

Stationary distribution π: [0.15384615 0.30769231 0.23076923 0.30769231]
Uniform? False

Row sums: [1. 1. 1. 1.]
Col sums: [0.58333333 1.58333333 0.75       1.08333333]
Doubly stochastic? False

Conditional entropies:
  Position 1: -0.000 bits
  Position 2: 1.000 bits
  Position 3: 1.585 bits
  Position 4: 2.000 bits


In [ ]:
graded_matrix_v1_counter, graded_matrix_v1_triplets = analyze_markov_design(graded_matrix_v1)
print(graded_matrix_v1_counter)
print(graded_matrix_v1_triplets)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

### **Summary Interpretation**

- Uniform unigram visitation: π = [0.15384615 0.30769231 0.23076923 0.30769231]
- Graded entropy at trigram level (excludes repetitions and trills):
```
Counter({np.float64(0.02): 3,
         np.float64(0.03): 5,
         np.float64(0.05): 5,
         np.float64(0.06): 1,
         np.float64(0.07): 1,
         np.float64(0.10): 4})
```
- Trigrams at each probability (excludeds repetitions and trills)
```
{0.02: [(4, 4, 2), (3, 4, 1), (3, 4, 4)],
 0.03: [(3, 4, 2), (4, 3, 2), (4, 3, 1), (4, 4, 1), (4, 4, 3)],
 0.05: [(2, 4, 4), (4, 2, 3), (3, 2, 4), (2, 4, 1), (2, 4, 3)],
 0.06: [(2, 3, 4)],
 0.07: [(2, 3, 1)],
 0.1: [(3, 1, 2), (1, 2, 4), (4, 1, 2), (1, 2, 3)]}
```
- Entropy at trigram level: 4.053 / 4.248 bits
- Entropy ratio (unpredictability): 0.95
- Uniform supported transition probability values on each row: True
- Data loss from repetitions & trills: ~22.9%

## 4-By-4 Optimizations

### 4-By-4 (Fixed Positioning)

In [ ]:
# MATRIX OPTIMIZATION

def optimize_doubly_stochastic(structure_mask):
    """
    Find doubly stochastic matrix for given structure

    Args:
        structure_mask: Boolean array indicating legal transitions

    Returns:
        P: Optimized transition matrix
    """
    n = structure_mask.shape[0]
    n_params = structure_mask.sum()

    # Map parameters to matrix
    param_to_idx = {}
    idx_to_param = []
    param_idx = 0

    for i in range(n):
        for j in range(n):
            if structure_mask[i,j]:
                param_to_idx[(i,j)] = param_idx
                idx_to_param.append((i,j))
                param_idx += 1

    def params_to_matrix(params):
        """Convert parameter vector to matrix"""
        P = np.zeros((n, n))
        for idx, (i, j) in enumerate(idx_to_param):
            P[i,j] = params[idx]
        return P

    def objective(params):
        """Minimize deviation from doubly stochastic"""
        P = params_to_matrix(params)
        col_sums = P.sum(axis=0)
        return np.sum((col_sums - 1.0)**2)

    # Row sum constraints
    constraints = []
    for i in range(n):
        row_indices = [param_to_idx[(i,j)] for j in range(n) if (i,j) in param_to_idx]

        def row_constraint(params, indices=row_indices):
            return np.sum([params[idx] for idx in indices]) - 1.0

        constraints.append({'type': 'eq', 'fun': row_constraint})

    # Bounds
    bounds = [(0, 1) for _ in range(n_params)]

    # Initial guess: uniform within row
    x0 = np.array([1.0/structure_mask[idx_to_param[i][0],:].sum()
                   for i in range(n_params)])

    # Optimize
    result = minimize(objective, x0, method='SLSQP',
                     bounds=bounds, constraints=constraints,
                     options={'maxiter': 1000})

    P_opt = params_to_matrix(result.x)

    return P_opt, result

In [ ]:
structure = np.array([
    [False, True, False, False],
    [False, False, True, True],
    [True, True, False, True],
    [True, True, True, True]
])

P_opt, result = optimize_doubly_stochastic(structure)
print("Optimized matrix:")
print(P_opt)
print("\nRow sums:", P_opt.sum(axis=1))
print("Col sums:", P_opt.sum(axis=0))

In [ ]:
graded_matrix_v2 = P_opt
graded_matrix_v2_display = display_matrix(graded_matrix_v2)
graded_matrix_v2_p_pi = get_stationary_distribution(graded_matrix_v2)

In [ ]:
graded_matrix_v2_counter, graded_matrix_v2_triplets = analyze_markov_design(graded_matrix_v2)

In [ ]:
graded_matrix_v2_counter
# graded_matrix_v2_triplets

### 4-By-4 (Flexible Positioning)

In [ ]:
print("="*70)
print("EXHAUSTIVE SEARCH: ALL 96 GRADED 4×4 STRUCTURES")
print("="*70)
print()

start_time = time.time()

# Generate all possible graded structures
def generate_all_structures():
    """Generate all 96 possible graded structures for 4×4"""
    structures = []

    # Position 0: Choose 1 from {0,1,2,3}
    for p0_targets in combinations(range(4), 1):
        # Position 1: Choose 2 from {0,1,2,3}
        for p1_targets in combinations(range(4), 2):
            # Position 2: Choose 3 from {0,1,2,3}
            for p2_targets in combinations(range(4), 3):
                # Position 3: All 4 (only one option)
                p3_targets = (0, 1, 2, 3)

                # Create structure
                structure = [
                    list(p0_targets),
                    list(p1_targets),
                    list(p2_targets),
                    list(p3_targets)
                ]
                structures.append(structure)

    return structures

print("Generating all structures...")
all_structures = generate_all_structures()
print(f"Generated {len(all_structures)} structures")
print()

# Create structure matrix
def structure_to_matrix(structure_list):
    """Convert structure list to boolean matrix"""
    matrix = np.zeros((4, 4), dtype=bool)
    for i in range(4):
        for j in structure_list[i]:
            matrix[i, j] = True
    return matrix

# Optimize probabilities for a given structure
def optimize_structure(structure_matrix, quick=True):
    legal_pos = [(i,j) for i in range(4) for j in range(4) if structure_matrix[i,j]]
    n_params = len(legal_pos)

    def params_to_matrix(params):
        P = np.zeros((4, 4))
        for idx, (i,j) in enumerate(legal_pos):
            P[i,j] = params[idx]
        return P

    def calc_stationary(P, max_iter=500):
        pi = np.ones(4) / 4
        for _ in range(max_iter):
            pi_new = pi @ P
            if np.allclose(pi, pi_new, atol=1e-6):
                break
            pi = pi_new
        return pi

    def calc_trigram_probs(P):
        pi = calc_stationary(P)
        trigrams = {}
        for i in range(4):
            for j in range(4):
                if P[i,j] > 1e-6:
                    for k in range(4):
                        if P[j,k] > 1e-6:
                            if not (i==j==k or i==k):
                                prob = pi[i] * P[i,j] * P[j,k]
                                if prob > 1e-10:
                                    trigrams[(i,j,k)] = prob
        return trigrams, pi

    def objective(params):
        P = params_to_matrix(params)

        # Doubly stochastic penalty
        col_dev = np.sum((P.sum(axis=0) - 1.0)**2)
        doubly_penalty = 1000 * col_dev

        # Calculate trigrams
        trigrams, pi = calc_trigram_probs(P)

        if len(trigrams) < 10:
            return 1e10

        # Evenness penalty
        sorted_probs = sorted(trigrams.values())
        n_levels = 5
        n_per_level = len(sorted_probs) / n_levels

        level_counts = []
        for lev in range(n_levels):
            start = int(lev * n_per_level)
            end = int((lev+1) * n_per_level)
            level_counts.append(end - start)

        target = len(sorted_probs) / n_levels
        evenness_penalty = 10 * np.sum([(c - target)**2 for c in level_counts])

        # Range penalty
        prob_range = max(sorted_probs) / (min(sorted_probs) + 1e-10)
        range_penalty = 0 if prob_range >= 4.0 else 50 * (4.0 - prob_range)**2

        # Entropy penalty
        probs_norm = np.array(sorted_probs) / np.sum(sorted_probs)
        entropy = -np.sum(probs_norm * np.log2(probs_norm + 1e-10))
        max_ent = np.log2(len(sorted_probs))
        ent_ratio = entropy / max_ent
        entropy_penalty = 50 * max(0, ent_ratio - 0.90)**2

        total = doubly_penalty + evenness_penalty + range_penalty + entropy_penalty
        return total

    # Constraints
    constraints = []
    for i in range(4):
        row_idx = [idx for idx, (r,c) in enumerate(legal_pos) if r==i]
        constraints.append({
            'type': 'eq',
            'fun': lambda p, ri=row_idx: np.sum([p[j] for j in ri]) - 1.0
        })

    bounds = [(0, 1) for _ in legal_pos]

    # Initial guess
    x0 = []
    for i in range(4):
        n_trans = structure_matrix[i,:].sum()
        for j in range(4):
            if structure_matrix[i,j]:
                x0.append(1.0/n_trans)

    # Optimize
    result = minimize(
        objective, x0, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'maxiter': 500 if quick else 2000, 'ftol': 1e-6}
    )

    P = params_to_matrix(result.x)
    trigrams, pi = calc_trigram_probs(P)

    # Calculate metrics
    col_sums = P.sum(axis=0)
    is_doubly = np.max(np.abs(col_sums - 1.0)) < 0.01

    sorted_probs = sorted(trigrams.values())
    prob_range = max(sorted_probs) / (min(sorted_probs) + 1e-10) if sorted_probs else 0

    probs_norm = np.array(sorted_probs) / np.sum(sorted_probs)
    entropy = -np.sum(probs_norm * np.log2(probs_norm + 1e-10))
    ent_ratio = entropy / np.log2(len(sorted_probs)) if sorted_probs else 1.0

    n_levels = 5
    level_counts = []
    for lev in range(n_levels):
        start = int(lev * len(sorted_probs) / n_levels)
        end = int((lev+1) * len(sorted_probs) / n_levels)
        level_counts.append(end - start)

    target_per = len(sorted_probs) / n_levels if sorted_probs else 0
    evenness = 1.0 - (max(level_counts) - min(level_counts)) / (target_per + 1e-10) if level_counts else 0

    metrics = {
        'n_trigrams': len(trigrams),
        'range': prob_range,
        'entropy_ratio': ent_ratio,
        'doubly_stochastic': is_doubly,
        'evenness': evenness,
        'level_counts': level_counts,
        'pi_deviation': np.max(np.abs(pi - 0.25)),
        'objective': result.fun
    }

    return P, result.fun, metrics

print("Searching all 96 structures...")
print("(This may take 1-2 minutes)")
print()

results = []

for idx, structure in enumerate(all_structures):
    if (idx + 1) % 20 == 0:
        print(f"  Processed {idx+1}/96...", end='\r')

    structure_matrix = structure_to_matrix(structure)
    P, score, metrics = optimize_structure(structure_matrix, quick=True)

    results.append({
        'structure': structure,
        'matrix': P,
        'score': score,
        'metrics': metrics
    })

print(f"  Processed 96/96... DONE!        ")
print()

elapsed = time.time() - start_time
print(f"Completed in {elapsed:.1f} seconds")
print()

# Sort by score
results_sorted = sorted(results, key=lambda x: x['score'])

print("="*70)
print("TOP 5 STRUCTURES")
print("="*70)
print()

for rank, result in enumerate(results_sorted[:5], 1):
    np.save(f"optimized_4x4_matrix_multistart_{rank}.npy", result['matrix'])
    m = result['metrics']
    print(f"Rank {rank}: Score {result['score']:.2f}")
    print(f"  Structure: {result['structure']}")
    print(f"  Trigrams: {m['n_trigrams']}, Range: {m['range']:.1f}:1, Entropy: {m['entropy_ratio']:.3f}")
    print(f"  Doubly: {m['doubly_stochastic']}, Evenness: {m['evenness']:.0%}, Dist: {m['level_counts']}")
    print()

print("="*70)
print("BEST STRUCTURE ANALYSIS")
print("="*70)
print()

best = results_sorted[0]
print(f"Structure: {best['structure']}")
print()

P_best = best['matrix']
print("Matrix:")
for i in range(4):
    row_vals = []
    for j in range(4):
        if best['structure'][i].__contains__(j):
            row_vals.append(f"{P_best[i,j]:.3f}")
        else:
            row_vals.append(" --- ")
    print(f"  [{', '.join(row_vals)}]")
print()

print(f"Col sums: {[f'{x:.3f}' for x in P_best.sum(axis=0)]}")
print()

m = best['metrics']
print("FINAL METRICS:")
print(f"  Trigrams: {m['n_trigrams']}")
print(f"  Range: {m['range']:.1f}:1")
print(f"  Entropy: {m['entropy_ratio']:.3f}")
print(f"  Doubly stochastic: {m['doubly_stochastic']}")
print(f"  Evenness: {m['evenness']:.0%}")
print(f"  Distribution: {m['level_counts']}")
print()

print("VERDICT:")
if m['evenness'] > 0.7 and m['entropy_ratio'] < 0.92:
    print("  ✓✓✓ FOUND FEASIBLE 4×4 STRUCTURE!")
elif m['evenness'] > 0.6:
    print("  ✓✓ Much better than original!")
else:
    print("  ✓ Better, but 4×4 remains challenging")

In [ ]:
graded_matrix_v3 = np.load("optimized_4x4_matrix_multistart_2.npy", allow_pickle=True)
graded_matrix_v3_display = display_matrix(graded_matrix_v3)
graded_matrix_v3_p_pi = get_stationary_distribution(graded_matrix_v3)

In [ ]:
graded_matrix_v3_counter, graded_matrix_v3_triplets = analyze_markov_design(graded_matrix_v3)

In [ ]:
graded_matrix_v3_counter
# graded_matrix_v3_triplets

## 6-By-6 Designs

In [ ]:
# SINGLE START
def create_initial_guess(n_states=6):
    """
    Create initial guess for optimization.

    Strategy: Start with structured doubly stochastic matrix that:
    - Has low self-loops (diagonal ≈ 0.05)
    - Favors adjacent transitions (circular topology)
    - Is doubly stochastic (balanced via Sinkhorn-Knopp)

    Args:
        n_states: Number of states (default 6)

    Returns:
        Flattened initial matrix vector
    """
    P = np.ones((n_states, n_states)) / n_states

    # Add circular structure: favor transitions to nearby states
    for i in range(n_states):
        for j in range(n_states):
            # Calculate circular distance
            dist = min(abs(i - j), n_states - abs(i - j))

            if dist == 0:
                P[i, j] = 0.05   # Low self-loops
            elif dist == 1:
                P[i, j] = 0.30   # High adjacent
            elif dist == 2:
                P[i, j] = 0.20   # Medium
            else:
                P[i, j] = 0.15   # Lower distant

    # Normalize rows
    P = P / P.sum(axis=1, keepdims=True)

    # Make doubly stochastic using Sinkhorn-Knopp algorithm
    # Iteratively normalize rows and columns until convergence
    for iteration in range(100):
        P = P / P.sum(axis=1, keepdims=True)  # Normalize rows
        P = P / P.sum(axis=0, keepdims=True)  # Normalize columns

    return P.flatten()

def optimize_6x6_matrix(verbose=True):
    """
    Main optimization function to find optimal 6×6 doubly stochastic matrix.

    Args:
        verbose: Whether to print progress (default True)

    Returns:
        P_opt: Optimized 6×6 transition matrix (numpy array)
        result: Full optimization result object from scipy.optimize
    """
    n_states = 6

    if verbose:
        print("="*70)
        print("OPTIMIZING 6×6 DOUBLY STOCHASTIC MATRIX")
        print("="*70)

    # Create constraints
    if verbose:
        print("\nCreating constraints...")
    constraints = create_constraints(n_states)
    if verbose:
        print(f"  {len(constraints)} constraints created")
        print(f"  - {n_states} row sum constraints")
        print(f"  - {n_states} column sum constraints")
        print(f"  - {n_states*n_states} non-negativity constraints")

    # Create initial guess
    if verbose:
        print("\nCreating initial guess...")
    x0 = create_initial_guess(n_states)
    if verbose:
        print(f"  Initial objective value: {objective_function(x0, n_states):.6f}")

    # Run optimization
    if verbose:
        print("\nRunning optimization...")
        print("  Method: SLSQP (Sequential Least Squares Programming)")
        print("  This may take 1-2 minutes...")

    result = minimize(
        fun=lambda x: objective_function(x, n_states),
        x0=x0,
        method='SLSQP',
        constraints=constraints,
        options={
            'maxiter': 1000,
            'ftol': 1e-9,
            'disp': verbose
        }
    )

    # Extract optimized matrix
    P_opt = result.x.reshape((n_states, n_states))

    # Clean up numerical precision errors
    P_opt = np.maximum(P_opt, 0)  # Clip negative values to 0
    P_opt = P_opt / P_opt.sum(axis=1, keepdims=True)  # Renormalize rows

    if verbose:
        print("\n" + "="*70)
        print("OPTIMIZATION COMPLETE")
        print("="*70)
        print(f"\nSuccess: {result.success}")
        print(f"Final objective value: {result.fun:.6f}")
        print(f"Number of iterations: {result.nit}")
        print(f"Function evaluations: {result.nfev}")

        print("\nOptimized Transition Matrix:")
        print(np.array2string(P_opt, precision=3, suppress_small=True))

        # Verify doubly stochastic
        print(f"\nVerification:")
        print(f"  Row sums: {np.array2string(P_opt.sum(axis=1), precision=6)}")
        print(f"  Col sums: {np.array2string(P_opt.sum(axis=0), precision=6)}")
        print(f"  All close to 1.0: {np.allclose(P_opt.sum(axis=1), 1.0) and np.allclose(P_opt.sum(axis=0), 1.0)}")

    np.save("optimized_6x6_matrix.npy", P_opt)
    return P_opt, result

In [ ]:
# MULTISTART
def calculate_stationary_distribution(P, method='pagerank'):
    """
    Calculate stationary distribution with robust fallback methods.

    Args:
        P: Transition matrix
        method: 'eigenvalue' or 'pagerank'

    Returns:
        pi: Stationary distribution, or None if calculation fails
    """
    n_states = len(P)

    # Check for degenerate matrix (columns/rows with all zeros)
    if np.any(P.sum(axis=0) < 1e-10) or np.any(P.sum(axis=1) < 1e-10):
        return None  # Degenerate matrix

    if method == 'eigenvalue':
        try:
            eigenvalues, eigenvectors = np.linalg.eig(P.T)
            stationary_idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-8)

            # Check if we actually found eigenvalue 1
            if np.abs(eigenvalues[stationary_idx] - 1.0) > 1e-6:
                return None

            pi = np.real(eigenvectors[:, stationary_idx])

            # Check for negative or invalid values
            if np.any(pi < -1e-10) or np.sum(pi) < 1e-10:
                return None

            pi = np.abs(pi)  # Handle numerical noise
            pi = pi / pi.sum()

            return pi
        except:
            return None

    elif method == 'pagerank':
        # PageRank method - more stable for doubly stochastic matrices
        # For doubly stochastic, stationary distribution should be uniform
        return np.ones(n_states) / n_states


def objective_function(x, n_states=6, sparsity_penalty=0.5):
    """
    Robust objective function with numerical safeguards and sparsity penalty.

    Args:
        x: Flattened matrix vector
        n_states: Number of states
        sparsity_penalty: Penalty weight for sparse matrices (higher = discourage sparsity)

    Returns:
        score: Objective value (lower is better)
    """
    P = x.reshape((n_states, n_states))

    # Check doubly stochastic
    row_sums = P.sum(axis=1)
    col_sums = P.sum(axis=0)
    if not (np.allclose(row_sums, 1.0, atol=1e-6) and np.allclose(col_sums, 1.0, atol=1e-6)):
        return 1e10  # Constraint violation

    # Check for degenerate structure (too sparse)
    n_zeros = np.sum(P < 1e-6)
    sparsity_ratio = n_zeros / (n_states * n_states)
    if sparsity_ratio > 0.5:  # More than 50% zeros
        return 1e10  # Reject overly sparse matrices

    # Calculate stationary distribution
    pi = calculate_stationary_distribution(P, method='pagerank')
    if pi is None:
        return 1e10  # Failed to calculate stationary distribution

    # Verify stationary distribution is valid
    if not np.allclose(np.dot(pi, P), pi, atol=1e-6):
        return 1e10  # Not a valid stationary distribution

    # Calculate trigram probabilities
    trigram_probs = {}
    for i, j, k in itertools.product(range(n_states), repeat=3):
        if i != j and j != k and i != k:
            prob = pi[i] * P[i, j] * P[j, k]
            if prob > 1e-10:  # Only include non-negligible probabilities
                trigram_probs[(i, j, k)] = prob

    # Check we have enough legal trigrams
    if len(trigram_probs) < 60:  # At least half of possible 120
        return 1e10  # Too few legal trigrams

    prob_values = np.array(list(trigram_probs.values()))
    n_legal = len(prob_values)
    retention = prob_values.sum()

    # Check data retention
    if retention < 0.6:  # At least 60% retention
        return 1e10  # Too much data loss

    # Calculate normalized entropy
    max_entropy = np.log(len(prob_values))
    actual_entropy = entropy(prob_values)

    # Check for numerical issues
    if not np.isfinite(actual_entropy):
        return 1e10

    normalized_entropy = actual_entropy / max_entropy if max_entropy > 0 else 1.0

    # Evaluate evenness of distribution across bins
    n_bins = 5
    prob_values_sorted = np.sort(prob_values)[::-1]
    bin_size = len(prob_values_sorted) // n_bins

    bin_counts = []
    for i in range(n_bins - 1):
        bin_counts.append(bin_size)
    bin_counts.append(len(prob_values_sorted) - (n_bins - 1) * bin_size)

    evenness = np.std(bin_counts) / np.mean(bin_counts) if np.mean(bin_counts) > 0 else 0.0

    # Add sparsity penalty (discourage matrices with many zeros)
    # Count entries < 0.01 (essentially zero)
    sparsity = np.sum(P < 0.01) / (n_states * n_states)

    # Combined objective
    max_legal_trigrams = n_states * (n_states - 1) * (n_states - 2)
    score = (
        1.0 * normalized_entropy +                     # Priority 1: Low entropy
        0.5 * evenness +                               # Priority 2: Even bins
        0.3 * (1.0 - retention) +                      # Priority 3: High retention
        0.2 * (1.0 - n_legal / max_legal_trigrams) +  # Priority 4: Many trigrams
        sparsity_penalty * sparsity                     # NEW: Penalty for sparsity
    )

    # Sanity check
    if not np.isfinite(score) or score < 0:
        return 1e10

    return score


def create_constraints(n_states=6, min_value=0.0):
    """
    Create constraints with optional minimum value for matrix entries.

    Args:
        n_states: Number of states
        min_value: Minimum value for each entry (default 0.0, recommend 0.01 to prevent sparsity)

    Returns:
        List of LinearConstraint objects
    """
    constraints = []
    n_params = n_states * n_states

    # Row sum constraints
    for i in range(n_states):
        A_row = np.zeros((1, n_params))
        for j in range(n_states):
            A_row[0, i * n_states + j] = 1.0
        constraints.append(LinearConstraint(A_row, [1.0], [1.0]))

    # Column sum constraints
    for j in range(n_states):
        A_col = np.zeros((1, n_params))
        for i in range(n_states):
            A_col[0, i * n_states + j] = 1.0
        constraints.append(LinearConstraint(A_col, [1.0], [1.0]))

    # Bounds: min_value <= P[i,j] <= 1.0
    # This prevents overly sparse matrices
    A_nonneg = np.eye(n_params)
    constraints.append(LinearConstraint(A_nonneg,
                                        np.full(n_params, min_value),
                                        np.ones(n_params)))

    return constraints


def create_random_doubly_stochastic(n_states=6, seed=None, min_value=0.01):
    """
    Create a random doubly stochastic matrix with minimum entry values.

    Args:
        n_states: Number of states
        seed: Random seed
        min_value: Minimum value for each entry (prevents extreme sparsity)

    Returns:
        Flattened doubly stochastic matrix
    """
    if seed is not None:
        np.random.seed(seed)

    # Start with random positive matrix, bounded away from zero
    P = np.random.rand(n_states, n_states) * (1.0 - min_value) + min_value

    # Make doubly stochastic via Sinkhorn-Knopp
    for iteration in range(200):  # More iterations for stability
        P = P / P.sum(axis=1, keepdims=True)
        P = P / P.sum(axis=0, keepdims=True)

        # Check convergence
        if iteration > 10 and np.allclose(P.sum(axis=0), 1.0, atol=1e-8) and \
           np.allclose(P.sum(axis=1), 1.0, atol=1e-8):
            break

    # Ensure minimum values are maintained
    P = np.maximum(P, min_value)
    P = P / P.sum(axis=1, keepdims=True)
    P = P / P.sum(axis=0, keepdims=True)

    return P.flatten()


def optimize_6x6_multistart(n_restarts=10, verbose=True, return_all=False,
                            sparsity_penalty=0.5, min_value=0.0):
    """
    Robust multi-start optimization with numerical safeguards.

    Args:
        n_restarts: Number of random starting points
        verbose: Print progress
        return_all: Return all results or just best
        sparsity_penalty: Penalty for sparse matrices (0.0-1.0, higher discourages sparsity)
        min_value: Minimum value for matrix entries (0.0-0.05, higher prevents sparsity)

    Returns:
        P_best, result_best (or all_results if return_all=True)
    """
    n_states = 6

    if verbose:
        print("="*70)
        print(f"ROBUST MULTI-START OPTIMIZATION ({n_restarts} restarts)")
        print("="*70)
        print(f"Settings:")
        print(f"  Sparsity penalty: {sparsity_penalty}")
        print(f"  Minimum entry value: {min_value}")

    # Create constraints
    constraints = create_constraints(n_states, min_value=min_value)

    # Store valid results only
    all_results = []
    failed_runs = 0

    for restart in range(n_restarts):
        if verbose:
            print(f"\n--- Restart {restart + 1}/{n_restarts} ---")

        try:
            # Create random initial guess
            x0 = create_random_doubly_stochastic(n_states, seed=restart, min_value=min_value)

            initial_score = objective_function(x0, n_states, sparsity_penalty)

            # Skip if initial guess is invalid
            if initial_score >= 1e9:
                if verbose:
                    print(f"Initial guess invalid, skipping...")
                failed_runs += 1
                continue

            if verbose:
                print(f"Initial score: {initial_score:.6f}")

            # Optimize
            result = minimize(
                fun=lambda x: objective_function(x, n_states, sparsity_penalty),
                x0=x0,
                method='SLSQP',
                constraints=constraints,
                options={
                    'maxiter': 1000,
                    'ftol': 1e-9,
                    'disp': False
                }
            )

            # Check if optimization succeeded
            if not result.success:
                if verbose:
                    print(f"Optimization failed: {result.message}")
                failed_runs += 1
                continue

            # Check if final score is valid
            if result.fun >= 1e9 or not np.isfinite(result.fun):
                if verbose:
                    print(f"Final solution invalid (score: {result.fun:.2e})")
                failed_runs += 1
                continue

            # Extract and clean matrix
            P = result.x.reshape((n_states, n_states))
            P = np.maximum(P, 0)
            P = P / P.sum(axis=1, keepdims=True)

            # Verify final matrix is valid
            final_score = objective_function(P.flatten(), n_states, sparsity_penalty)
            if final_score >= 1e9:
                if verbose:
                    print(f"Final matrix validation failed")
                failed_runs += 1
                continue

            if verbose:
                print(f"Final score: {final_score:.6f}")
                print(f"Success: {result.success}")
                print(f"Improvement: {((initial_score - final_score) / initial_score * 100):.1f}%")

                # Report sparsity
                sparsity = np.sum(P < 0.01) / (n_states * n_states)
                print(f"Sparsity: {sparsity*100:.1f}% (entries < 0.01)")

            all_results.append((P, result, final_score))

        except Exception as e:
            if verbose:
                print(f"Exception: {e}")
            failed_runs += 1
            continue

    # Check if we got any valid results
    if len(all_results) == 0:
        raise RuntimeError(f"All {n_restarts} optimization runs failed! "
                         f"Try adjusting sparsity_penalty or min_value.")

    # Sort by score
    all_results.sort(key=lambda x: x[2])

    if verbose:
        print("\n" + "="*70)
        print("MULTI-START OPTIMIZATION COMPLETE")
        print("="*70)
        print(f"\nResults summary:")
        print(f"  Successful runs: {len(all_results)}/{n_restarts}")
        print(f"  Failed runs: {failed_runs}")
        print(f"  Best score:  {all_results[0][2]:.6f}")
        print(f"  Worst score: {all_results[-1][2]:.6f}")
        print(f"  Range:       {all_results[-1][2] - all_results[0][2]:.6f}")
        print(f"  Std dev:     {np.std([r[2] for r in all_results]):.6f}")

        # Check uniqueness
        unique_scores = len(set(np.round([r[2] for r in all_results], 4)))
        print(f"\n  Unique solutions found: {unique_scores}")

        if unique_scores == 1:
            print("  ✓ All restarts converged to same optimum")
            print("    → High confidence this is the global optimum")
        elif unique_scores <= 3:
            print("  ✓ Few distinct solutions found")
            print("    → Likely found global optimum")
        else:
            print("  ⚠ Multiple local minima detected")
            print("    → Consider more restarts or adjusting parameters")

        print("\n  Top 5 scores:")
        for i, (P, result, score) in enumerate(all_results[:min(5, len(all_results))], 1):
            sparsity = np.sum(P < 0.01) / (n_states * n_states)
            print(f"    {i}. Score: {score:.6f}, Sparsity: {sparsity*100:.1f}%")

    if return_all:
        return all_results
    else:
        P_best, result_best, score_best = all_results[0]

        if verbose:
            print("\nBest Matrix:")
            print(np.array2string(P_best, precision=3, suppress_small=True))

            # Calculate and display statistics
            pi = calculate_stationary_distribution(P_best, method='pagerank')
            print(f"\nStationary distribution:")
            print(np.array2string(pi, precision=4))

        return P_best, result_best


# Example usage
if __name__ == "__main__":
    # Recommended settings for robust optimization:
    # - sparsity_penalty=0.5: Moderate penalty on sparse matrices
    # - min_value=0.0: No hard minimum (penalty handles it)
    # - n_restarts=20: Good balance of thoroughness and speed

    print("Running robust multi-start optimization...\n")

    P_optimal, result = optimize_6x6_multistart(
        n_restarts=20,
        verbose=True,
        sparsity_penalty=0.5,
        min_value=0
    )

    # Save
    np.save('optimized_6x6_matrix_robust.npy', P_optimal)
    print("\n✓ Matrix saved to: optimized_6x6_matrix_robust.npy")


In [ ]:
graded_matrix_v4 = np.load("optimized_6x6_matrix_robust.npy")
graded_matrix_v4_display = display_matrix(graded_matrix_v4)
graded_matrix_v4_p_pi = get_stationary_distribution(graded_matrix_v4)

In [ ]:
graded_matrix_v4_counter, graded_matrix_v4_triplets = analyze_markov_design(graded_matrix_v4)

In [ ]:
graded_matrix_v4_counter

## General Optimizer Function by Conditional Entropy

In [ ]:
"""
Systematic matrix optimizer for position-level conditional entropy gradients.

Optimizes transition matrices of sizes 4-8 to achieve:
1. Graded conditional entropy across positions
2. Maximal spread and even spacing of entropy levels
3. Doubly stochastic constraint (with small tolerance)
4. Target number of distinct entropy levels = matrix size (or size-1)
"""

def calculate_conditional_entropy(P, position):
    """
    Calculate conditional entropy H(X_{t+1} | X_t = position).

    This is the entropy of the transition distribution from a given position.
    H(X|position) = -sum_j P[position,j] * log2(P[position,j])

    Args:
        P: Transition matrix
        position: Which position to calculate entropy for

    Returns:
        Conditional entropy in bits
    """
    probs = P[position]
    # Filter out zero probabilities
    probs = probs[probs > 1e-10]
    if len(probs) == 0:
        return 0.0
    return scipy_entropy(probs, base=2)


def calculate_all_conditional_entropies(P):
    """Calculate conditional entropy for each position."""
    n_states = len(P)
    entropies = []
    for i in range(n_states):
        entropies.append(calculate_conditional_entropy(P, i))
    return np.array(entropies)


def objective_function_conditional_entropy(x, n_states, target_profile='even_spread'):
    """
    Objective function that optimizes for graded conditional entropy.

    Goals:
    1. Achieve target entropy profile (e.g., evenly spaced levels)
    2. Maximize spread (distance between min and max entropy)
    3. Minimize clustering (want distinct levels)
    4. Maintain doubly stochastic

    Args:
        x: Flattened matrix vector
        n_states: Matrix size
        target_profile: 'even_spread', 'linear', 'exponential', or array of target entropies

    Returns:
        score: Combined objective (lower is better)
    """
    P = x.reshape((n_states, n_states))

    # Verify doubly stochastic (should be satisfied by constraints)
    row_sums = P.sum(axis=1)
    col_sums = P.sum(axis=0)
    if not (np.allclose(row_sums, 1.0, atol=1e-4) and np.allclose(col_sums, 1.0, atol=1e-4)):
        return 1e10

    # Calculate conditional entropies for each position
    entropies = calculate_all_conditional_entropies(P)

    # Sort entropies to analyze distribution
    sorted_entropies = np.sort(entropies)

    # Calculate target entropies based on profile
    max_entropy = np.log2(n_states)  # Maximum possible entropy

    if isinstance(target_profile, str):
        if target_profile == 'even_spread':
            # Evenly spaced from 0 to max_entropy
            target_entropies = np.linspace(0, max_entropy, n_states)
        elif target_profile == 'linear':
            # Linear gradient (same as even_spread)
            target_entropies = np.linspace(0, max_entropy, n_states)
        elif target_profile == 'exponential':
            # Exponential gradient (bigger steps at higher entropies)
            target_entropies = np.array([max_entropy * (2**i - 1) / (2**n_states - 1)
                                         for i in range(n_states)])
        else:
            raise ValueError(f"Unknown profile: {target_profile}")
    else:
        # Custom target array provided
        target_entropies = np.array(target_profile)

    # Ensure target_entropies are sorted
    target_entropies = np.sort(target_entropies)

    # Calculate metrics

    # 1. Distance from target profile
    profile_distance = np.sum((sorted_entropies - target_entropies)**2)

    # 2. Spread (want large range)
    spread = sorted_entropies[-1] - sorted_entropies[0]
    spread_penalty = (max_entropy - spread)**2  # Penalize small spread

    # 3. Evenness (want equal spacing between adjacent levels)
    if len(sorted_entropies) > 1:
        gaps = np.diff(sorted_entropies)
        gap_variance = np.var(gaps)  # Low variance = even spacing
    else:
        gap_variance = 0

    # 4. Distinctness (penalize clustering)
    # Count pairs of entropies that are too close together
    distinctness_threshold = 0.1  # Entropies should differ by at least 0.1 bits
    n_close_pairs = 0
    for i in range(len(sorted_entropies) - 1):
        if sorted_entropies[i+1] - sorted_entropies[i] < distinctness_threshold:
            n_close_pairs += 1
    distinctness_penalty = n_close_pairs * 10.0

    # Combined objective (minimize)
    score = (
        2.0 * profile_distance +      # Priority 1: Match target profile
        1.0 * spread_penalty +         # Priority 2: Maximize spread
        0.5 * gap_variance +           # Priority 3: Even spacing
        1.0 * distinctness_penalty     # Priority 4: Distinct levels
    )

    return score


def create_constraints(n_states):
    """
    Create linear constraints for doubly stochastic matrix.
    Allows small tolerance for numerical stability.
    """
    constraints = []
    n_params = n_states * n_states
    tolerance = 1e-4  # Small tolerance for "almost" doubly stochastic

    # Row sum constraints: sum_j P[i,j] = 1 for all i
    for i in range(n_states):
        A_row = np.zeros((1, n_params))
        for j in range(n_states):
            A_row[0, i * n_states + j] = 1.0
        constraints.append(LinearConstraint(A_row, [1.0 - tolerance], [1.0 + tolerance]))

    # Column sum constraints: sum_i P[i,j] = 1 for all j
    for j in range(n_states):
        A_col = np.zeros((1, n_params))
        for i in range(n_states):
            A_col[0, i * n_states + j] = 1.0
        constraints.append(LinearConstraint(A_col, [1.0 - tolerance], [1.0 + tolerance]))

    # Non-negativity: P[i,j] >= 0
    A_nonneg = np.eye(n_params)
    constraints.append(LinearConstraint(A_nonneg, np.zeros(n_params), np.ones(n_params)))

    return constraints


def create_initial_guess(n_states, target_profile='even_spread'):
    """
    Create initial guess with approximate target entropy profile.

    Strategy:
    - For low-entropy positions: Make sparse (few non-zero transitions)
    - For high-entropy positions: Make dense (many transitions)
    - Balance to maintain doubly stochastic
    """
    P = np.zeros((n_states, n_states))

    # Determine target entropy for each position
    max_entropy = np.log2(n_states)
    if target_profile == 'even_spread' or target_profile == 'linear':
        target_entropies = np.linspace(0, max_entropy, n_states)
    elif target_profile == 'exponential':
        target_entropies = np.array([max_entropy * (2**i - 1) / (2**n_states - 1)
                                     for i in range(n_states)])
    else:
        target_entropies = np.array(target_profile)

    # For each position, create transition distribution with appropriate entropy
    for i in range(n_states):
        target_H = target_entropies[i]

        if target_H < 0.1:
            # Very low entropy: deterministic transition
            j = (i + 1) % n_states  # Transition to next state
            P[i, j] = 1.0
        elif target_H < max_entropy * 0.3:
            # Low entropy: concentrate on 2 states
            j1 = (i + 1) % n_states
            j2 = (i + 2) % n_states
            p1 = 0.8
            p2 = 0.2
            P[i, j1] = p1
            P[i, j2] = p2
        elif target_H < max_entropy * 0.7:
            # Medium entropy: spread over 3-4 states
            n_nonzero = n_states // 2
            indices = [(i + k + 1) % n_states for k in range(n_nonzero)]
            probs = np.random.dirichlet(np.ones(n_nonzero))
            for j, prob in zip(indices, probs):
                P[i, j] = prob
        else:
            # High entropy: spread over all states
            # Use Dirichlet to get random distribution
            probs = np.random.dirichlet(np.ones(n_states))
            P[i, :] = probs

    # Make doubly stochastic using Sinkhorn-Knopp algorithm
    for iteration in range(200):
        P = P / P.sum(axis=1, keepdims=True)  # Normalize rows
        P = P / P.sum(axis=0, keepdims=True)  # Normalize columns

        # Check convergence
        row_sums = P.sum(axis=1)
        col_sums = P.sum(axis=0)
        if np.allclose(row_sums, 1.0, atol=1e-6) and np.allclose(col_sums, 1.0, atol=1e-6):
            break

    return P.flatten()


def optimize_matrix(n_states, target_profile='even_spread', n_attempts=3, verbose=True):
    """
    Optimize transition matrix for graded conditional entropy.

    Args:
        n_states: Matrix size (4-8)
        target_profile: 'even_spread', 'linear', 'exponential', or custom array
        n_attempts: Number of random restarts
        verbose: Print progress

    Returns:
        best_P: Best optimized matrix
        best_result: Optimization result
        best_entropies: Conditional entropies of best matrix
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"OPTIMIZING {n_states}×{n_states} MATRIX")
        print(f"Target profile: {target_profile}")
        print(f"{'='*70}")

    constraints = create_constraints(n_states)

    best_score = np.inf
    best_P = None
    best_result = None

    for attempt in range(n_attempts):
        if verbose:
            print(f"\nAttempt {attempt + 1}/{n_attempts}...")

        # Create initial guess
        x0 = create_initial_guess(n_states, target_profile)

        if verbose:
            P0 = x0.reshape((n_states, n_states))
            entropies0 = calculate_all_conditional_entropies(P0)
            print(f"  Initial entropies: {np.array2string(np.sort(entropies0), precision=3)}")
            print(f"  Initial objective: {objective_function_conditional_entropy(x0, n_states, target_profile):.6f}")

        # Run optimization
        result = minimize(
            fun=lambda x: objective_function_conditional_entropy(x, n_states, target_profile),
            x0=x0,
            method='SLSQP',
            constraints=constraints,
            options={
                'maxiter': 1000,
                'ftol': 1e-9,
                'disp': False
            }
        )

        if result.success and result.fun < best_score:
            best_score = result.fun
            best_P = result.x.reshape((n_states, n_states))
            best_result = result

            if verbose:
                entropies = calculate_all_conditional_entropies(best_P)
                print(f"  ✓ Success! Objective: {result.fun:.6f}")
                print(f"  Final entropies: {np.array2string(np.sort(entropies), precision=3)}")

    # Clean up best matrix
    if best_P is not None:
        best_P = np.maximum(best_P, 0)  # Clip negatives
        best_P = best_P / best_P.sum(axis=1, keepdims=True)  # Renormalize rows

        # Final Sinkhorn-Knopp to ensure doubly stochastic
        for _ in range(100):
            best_P = best_P / best_P.sum(axis=1, keepdims=True)
            best_P = best_P / best_P.sum(axis=0, keepdims=True)

    best_entropies = calculate_all_conditional_entropies(best_P) if best_P is not None else None

    if verbose:
        print(f"\n{'='*70}")
        print(f"OPTIMIZATION COMPLETE")
        print(f"{'='*70}")
        if best_P is not None:
            print(f"Best objective: {best_score:.6f}")
            print(f"Entropy range: [{best_entropies.min():.3f}, {best_entropies.max():.3f}] bits")
            print(f"Entropy spread: {best_entropies.max() - best_entropies.min():.3f} bits")
            sorted_entropies = np.sort(best_entropies)
            if len(sorted_entropies) > 1:
                gaps = np.diff(sorted_entropies)
                print(f"Mean gap: {gaps.mean():.3f} bits (SD: {gaps.std():.3f})")
        else:
            print("⚠ Optimization failed!")

    return best_P, best_result, best_entropies


def analyze_matrix(P, n_states, save_path=None):
    """Comprehensive analysis of optimized matrix."""
    entropies = calculate_all_conditional_entropies(P)
    sorted_entropies = np.sort(entropies)

    analysis = {
        'n_states': n_states,
        'matrix': P.tolist(),
        'entropies': entropies.tolist(),
        'sorted_entropies': sorted_entropies.tolist(),
        'statistics': {
            'min': float(entropies.min()),
            'max': float(entropies.max()),
            'mean': float(entropies.mean()),
            'std': float(entropies.std()),
            'range': float(entropies.max() - entropies.min()),
        },
        'row_sums': P.sum(axis=1).tolist(),
        'col_sums': P.sum(axis=0).tolist(),
        'doubly_stochastic': bool(
            np.allclose(P.sum(axis=1), 1.0, atol=1e-3) and
            np.allclose(P.sum(axis=0), 1.0, atol=1e-3)
        ),
        'sparsity': float(np.sum(P < 1e-6) / P.size),  # Proportion of near-zero elements
    }

    # Gap analysis
    if len(sorted_entropies) > 1:
        gaps = np.diff(sorted_entropies)
        analysis['gap_statistics'] = {
            'mean': float(gaps.mean()),
            'std': float(gaps.std()),
            'min': float(gaps.min()),
            'max': float(gaps.max()),
            'gaps': gaps.tolist()
        }

    # Count distinct entropy levels (threshold = 0.05 bits)
    threshold = 0.05
    distinct_levels = 1
    for i in range(1, len(sorted_entropies)):
        if sorted_entropies[i] - sorted_entropies[i-1] > threshold:
            distinct_levels += 1
    analysis['distinct_levels'] = distinct_levels

    if save_path:
        with open(save_path, 'w') as f:
            json.dump(analysis, f, indent=2)

    return analysis


def print_matrix_summary(P, n_states):
    """Print formatted summary of matrix."""
    entropies = calculate_all_conditional_entropies(P)
    sorted_entropies = np.sort(entropies)

    print(f"\nTransition Matrix ({n_states}×{n_states}):")
    print("       " + "".join([f"To {i:2d}    " for i in range(n_states)]))
    for i in range(n_states):
        row_str = f"From {i}  "
        for j in range(n_states):
            row_str += f"{P[i,j]:7.4f}  "
        print(row_str)

    print(f"\nConditional Entropies H(X_{{t+1}} | X_t):")
    for i in range(n_states):
        print(f"  Position {i}: {entropies[i]:.3f} bits")

    print(f"\nSorted Entropies:")
    for i, h in enumerate(sorted_entropies):
        print(f"  Level {i+1}: {h:.3f} bits")

    if len(sorted_entropies) > 1:
        gaps = np.diff(sorted_entropies)
        print(f"\nGaps between levels:")
        for i, gap in enumerate(gaps):
            print(f"  {i+1}→{i+2}: {gap:.3f} bits")

    print(f"\nStatistics:")
    print(f"  Range: [{sorted_entropies.min():.3f}, {sorted_entropies.max():.3f}] bits")
    print(f"  Spread: {sorted_entropies.max() - sorted_entropies.min():.3f} bits")
    print(f"  Mean: {entropies.mean():.3f} ± {entropies.std():.3f} bits")
    print(f"  Sparsity: {np.sum(P < 1e-6) / P.size * 100:.1f}% zeros")

    print(f"\nDoubly Stochastic Check:")
    print(f"  Row sums: {np.array2string(P.sum(axis=1), precision=6)}")
    print(f"  Col sums: {np.array2string(P.sum(axis=0), precision=6)}")
    print(f"  Status: {'✓ Valid' if np.allclose(P.sum(axis=1), 1.0, atol=1e-3) and np.allclose(P.sum(axis=0), 1.0, atol=1e-3) else '✗ Invalid'}")


def optimize_all_sizes(sizes=[4, 5, 6, 7, 8], profile='even_spread', output_dir='optimization_results'):
    """
    Optimize matrices for all requested sizes.

    Args:
        sizes: List of matrix sizes to optimize
        profile: Target entropy profile
        output_dir: Where to save results

    Returns:
        results: Dictionary mapping size to (matrix, entropies, analysis)
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    results = {}

    print("="*70)
    print("SYSTEMATIC MATRIX OPTIMIZATION")
    print("="*70)
    print(f"Sizes: {sizes}")
    print(f"Profile: {profile}")
    print(f"Output directory: {output_dir}")

    for n in sizes:
        print(f"\n{'#'*70}")
        print(f"# SIZE {n}×{n}")
        print(f"{'#'*70}")

        # Optimize
        P, result, entropies = optimize_matrix(n, profile, n_attempts=5, verbose=True)

        if P is not None:
            P_sorted = sort_by_entropy(P)
            # Print summary
            print_matrix_summary(P_sorted, n)

            # Analyze
            analysis = analyze_matrix(P_sorted, n, save_path=output_path / f'analysis_{n}x{n}.json')

            # Save matrix
            np.save(output_path / f'matrix_{n}x{n}.npy', P_sorted)

            # Save results
            results[n] = {
                'matrix': P_sorted,
                'entropies': entropies,
                'analysis': analysis
            }

            print(f"\n✓ Saved to {output_path}/matrix_{n}x{n}.npy")
        else:
            print(f"\n✗ Failed to optimize {n}×{n} matrix")

    # Create summary comparison
    print("\n" + "="*70)
    print("SUMMARY COMPARISON")
    print("="*70)
    print(f"\n{'Size':<6} {'Min H':<8} {'Max H':<8} {'Spread':<8} {'Mean Gap':<10} {'Distinct':<10} {'Sparsity':<10}")
    print("-"*70)

    for n in sorted(results.keys()):
        res = results[n]
        stats = res['analysis']['statistics']
        gap_stats = res['analysis'].get('gap_statistics', {})
        distinct = res['analysis']['distinct_levels']
        sparsity = res['analysis']['sparsity']

        print(f"{n}×{n:<4} {stats['min']:<8.3f} {stats['max']:<8.3f} {stats['range']:<8.3f} "
              f"{gap_stats.get('mean', 0):<10.3f} {distinct:<10d} {sparsity*100:<9.1f}%")

    return results


if __name__ == "__main__":
    # Run optimization for all sizes
    results = optimize_all_sizes(
        sizes=[4, 5, 6, 7, 8],
        profile='even_spread',
        output_dir='graded_entropy_matrices'
    )

    print("\n" + "="*70)
    print("ALL OPTIMIZATIONS COMPLETE")
    print("="*70)
    print(f"\nResults saved to: graded_entropy_matrices/")
    print(f"Files generated:")
    print(f"  - matrix_NxN.npy (transition matrices)")
    print(f"  - analysis_NxN.json (detailed statistics)")

In [ ]:
analyze_markov_design(np.load("graded_entropy_matrices/matrix_4x4.npy"))

In [ ]:
analyze_markov_design(np.load("graded_entropy_matrices/matrix_5x5.npy"))

In [ ]:
analyze_markov_design(np.load("graded_entropy_matrices/matrix_6x6.npy"))

In [ ]:
analyze_markov_design(np.load("graded_entropy_matrices/matrix_7x7.npy"))

In [ ]:
analyze_markov_design(np.load("graded_entropy_matrices/matrix_8x8.npy"))

In [ ]:
analyze_markov_design(sequence=[1,3,0,5,3,4,2,0,5,1,3,3,1,5,4,3,4,2,0,5,1,3,4,2,0,3,1,5,3,4,2,0,3,4,2,0,1,1,5,4,2,0,1,5,1,3,4,3,4,2,0,1,5,4,2,0,5,3,3,3,3,4,2,0,3,1,5,4,2,0,1,5,1,3,4,2,0,4,2,0,5,3,4,2,0,5,4,2,0,1,5,3,4,2,0,5,4,2,0,5,4,2,0,3,1,1,3,4,3,1,5,4,3,4,2,0,4,2,0,5,3,1,5,1,5,2,0,3,1,1,3,4,2,0,5,4,2,0,5,4,2,0,5,1,5,2,0,5,2,0,1,5,1,5,1,3,4,2,0,5,4,2,0,3,4,2,0,5,4,2,0,5,4,2,0,1,5,3,1,3,4,3,3,1,1,5,3,1,5,3,3,4,2,0,5,4,3,4,2,0,4,2,0,5,4,2,0,5,2,0,5,4,3,4,2,0,5,1,1,1,4,2,0,4,2,0,1,5,2,0,3,4,3,4,3,1,5,1,1,5,1,4,2,0,5,3,3,3,4,2,0,5,4,2,0,1,5,2,0,1,1,3,3,1,5,2,0,5,3,1,1,1,1,1,1,1,1,1,1,5,3,4,2,0,5,4,2,0,5,4,2,0,1,3,1,5,4,2,0,3,4,2,0,5,3,1,5,1,1,3,3,4,2,0,5,4,3,4,2,0,1,5,1,1,3,1,1,3,3,4,2,0,5,2,0,5,2,0,3,5,2,0,5,4,2,0,5,3,4,2,0,5,4,2,0,5,1,5,1,5,1,1,1,3,1,1,1,5,2,0,5,4,2,0,1,4,2,0,3,4,2,0,5,4,2,0,5,4,2,0,5,1,1,1,3,1,5,2,0,1,3,4,2,0,4,2,0,5,1,5,1,1,4,2,0,3,1,1,1,1])


1-GRAM DISTRIBUTION SUMMARY

🚫 Exclusions:
  Original count:         420
  Repetitions:            0
  Reptitions Type Count:  0
  Trills:                 0
  Trills Type Count:      0
  Total excluded:         0 (0.0%)
  Remaining:              420

📊 Basic Stats:
  Total 1-grams:  420
  Unique types:       6
  Entropy:            2.579 bits (max: 2.585)
  Entropy Ratio:      0.9977933188678548
  KL Divergence:      0.0057041879776442705

📈 Probability Distribution:
  Mean:               0.166667
  Median:             0.163095
  Std dev:            0.015058
  Min:                0.150000
  Max:                0.197619
  Range (max/min):    1.3x

🎯 Distribution Shape:
  Very high (>75%ile): 2 types
  High (50-75%ile):    1 types
  Medium (25-50%ile):  1 types
  Low (<25%ile):       2 types

2-GRAM DISTRIBUTION SUMMARY

🚫 Exclusions:
  Original count:         419
  Repetitions:            0
  Reptitions Type Count:  0
  Trills:                 0
  Trills Type Count:      0
  Total excl

(Counter({np.float64(0.0): 7,
          np.float64(0.01): 18,
          np.float64(0.02): 11,
          np.float64(0.03): 3,
          np.float64(0.04): 1,
          np.float64(0.05): 1,
          np.float64(0.06): 1,
          np.float64(0.07): 1,
          np.float64(0.1): 1,
          np.float64(0.15): 1}),
 {0.0: [(1, 3, 0),
   (3, 0, 5),
   (4, 3, 3),
   (5, 1, 4),
   (0, 3, 5),
   (3, 5, 2),
   (0, 1, 4)],
  0.01: [(5, 1, 3),
   (1, 3, 3),
   (3, 3, 1),
   (5, 4, 3),
   (0, 3, 1),
   (0, 3, 4),
   (0, 1, 1),
   (1, 1, 5),
   (5, 3, 3),
   (3, 3, 4),
   (2, 0, 4),
   (0, 4, 2),
   (4, 3, 1),
   (5, 3, 1),
   (0, 5, 2),
   (1, 1, 4),
   (1, 4, 2),
   (0, 1, 3)],
  0.02: [(0, 5, 3),
   (5, 3, 4),
   (0, 5, 1),
   (1, 5, 4),
   (1, 3, 4),
   (1, 5, 3),
   (0, 1, 5),
   (3, 1, 1),
   (1, 1, 3),
   (1, 5, 2),
   (5, 1, 1)],
  0.03: [(3, 1, 5), (2, 0, 3), (5, 2, 0)],
  0.04: [(2, 0, 1)],
  0.05: [(0, 5, 4)],
  0.06: [(5, 4, 2)],
  0.07: [(3, 4, 2)],
  0.1: [(2, 0, 5)],
  0.15: [(4, 2, 0

In [ ]:
[1,3,0,5,3,4,2,0,5,1,3,3,1,5,4,3,4,2,0,5,1,3,4,2,0,3,1,5,3,4,2,0,3,4,2,0,1,1,5,4,2,0,1,5,1,3,4,3,4,2,0,1,5,4,2,0,5,3,3,3,3,4,2,0,3,1,5,4,2,0,1,5,1,3,4,2,0,4,2,0,5,3,4,2,0,5,4,2,0,1,5,3,4,2,0,5,4,2,0,5,4,2,0,3,1,1,3,4,3,1,5,4,3,4,2,0,4,2,0,5,3,1,5,1,5,2,0,3,1,1,3,4,2,0,5,4,2,0,5,4,2,0,5,1,5,2,0,5,2,0,1,5,1,5,1,3,4,2,0,5,4,2,0,3,4,2,0,5,4,2,0,5,4,2,0,1,5,3,1,3,4,3,3,1,1,5,3,1,5,3,3,4,2,0,5,4,3,4,2,0,4,2,0,5,4,2,0,5,2,0,5,4,3,4,2,0,5,1,1,1,4,2,0,4,2,0,1,5,2,0,3,4,3,4,3,1,5,1,1,5,1,4,2,0,5,3,3,3,4,2,0,5,4,2,0,1,5,2,0,1,1,3,3,1,5,2,0,5,3,1,1,1,1,1,1,1,1,1,1,5,3,4,2,0,5,4,2,0,5,4,2,0,1,3,1,5,4,2,0,3,4,2,0,5,3,1,5,1,1,3,3,4,2,0,5,4,3,4,2,0,1,5,1,1,3,1,1,3,3,4,2,0,5,2,0,5,2,0,3,5,2,0,5,4,2,0,5,3,4,2,0,5,4,2,0,5,1,5,1,5,1,1,1,3,1,1,1,5,2,0,5,4,2,0,1,4,2,0,3,4,2,0,5,4,2,0,5,4,2,0,5,1,1,1,3,1,5,2,0,1,3,4,2,0,4,2,0,5,1,5,1,1,4,2,0,3,1,1,1,1]
[2,0,3,3,2,0,3,2,0,1,2,0,1,2,0,3,1,2,0,0,1,2,0,3,1,1,3,3,3,2,0,3,2,0,3,2,0,1,2,0,1,3,1,2,0,3,1,3,1,1,2,0,3,1,2,0,0,1,2,0,1,2,0,1,2,0,1,2,0,3,1,1,2,0,3,3,3,3,3,1,3,2,0,1,2,0,3,3,1,2,0,1,2,0,1,2,0,1,2,0,1,2,0,1,3,2,0,1,3,1,2,0,1,2,0,3,1,2,0,3,1,2,0,3,2,0,3,3,1,2,0,1,3,1,1,2,0,1,3,2,0,3,1,3,1,3,3,2,0,1,3,1,3,1,2,0,1,2,0,3,2,0,1,1,2,0,1,2,0,1,2,0,1,2,0,3,1,3,2,0,1,3,3,2,0,1,2,0,1,3,3,3,3,3,1,3,3,3,2,0,1,2,0,3,3,1,2,0,3,1,1,2,0,3,1,2,0,1,2,0,1,3,3,2,0,1,2,0,3,3,1,2,0,3,2,0,1,2,0,3,1,2,0,1,2,0,1,2,0,1,2,0,1,2,0,1,3,2,0,1,2,0,3,3,2,0,1,3,3,1,2,0,3,3,1,3,3,3,2,0]
[3,0,6,5,3,0,6,5,3,2,0,4,1,5,3,1,5,3,2,0,0,0,6,2,6,4,2,1,5,3,4,4,0,6,4,4,0,4,2,6,2,0,6,0,6,0,4,1,5,3,2,0,6,1,5,3,1,0,4,1,5,3,1,5,3,0,6,0,6,1,0,6,1,0,6,4,1,0,0,6,0,6,1,5,3,4,2,0,0,0,4,1,5,3,2,1,5,3,2,0,0,0,6,5,3,2,1,5,3,4,0,6,4,1,5,3,0,6,5,3,2,0,0,6,1,5,3,2,0,6,0,6,0,6,0,6,4,4,1,5,3,2,0,6,1,5,3,2,1,5,3,4,0,6,1,5,3,2,0,6,0,4,4,0,0,6,5,3,2,4,4,0,0,6,4,2,1,5,3,2,1,5,3,2,4,4,1,5,3,2,0,6,0,4,4,4,1,5,3,4,4,1,5,3,4,2,4,0,6,4,1,5,3,2,0,6,1,5,3,1,0,6,1,5,3,2,6,0,6,2,6,1,5,3,0,4,4,2,1,5,3,4,2,4,4,1,5,3,2,4,1,5,3,4,2,1,5,3,2,4,1,0,6,4,0,6,1,5,3,4,2,6,1,5,3,4,1,5,3,2,4,2,0,6,5,3,2,4,0,6,5,3,4,2,6,5,3,0,6,1,5,3,1,5,3,2,6,1,5,3,2,6,4,0,6,1,5,3,0,6,5,3,2,0,0,6,1,0,4,1,5,3,2,4,2,1,5,3,2,0,6,1,5,3,2,0,6,5,3,4,4,0,6,4,1,5,3,1,5,3,2,4,1,5,3,2,4,0,6,1,5,0,6,1,5,3,4,1,5,3,2,1,5,3,2,0,6,1,5,3,2,6,4,2,6,3,2,1,5,3,2,4,2,1,5,3,1,5,3,2,0,6,5,3,2,6,1,5,3,1,5,3,2,1,5,3,2,6,1,5,3,2,1,5,3,1,5,3,4,4,2,1,5,3,2,0,4,2,4,1,5,3,0,4,4,2,0,6,4,2,1,5,3,0,0,6,1,5,3,4,1,5,3,2,4,2,1,5,3,2,0,6,0,6,0,4,4,1,5,3]
[2,5,3,4,1,2,0,3,0,3,4,3,4,1,2,0,5,0,4,1,2,0,5,4,1,2,0,3,4,1,2,5,4,3,4,1,2,5,0,5,0,0,5,4,1,2,5,3,3,3,0,0,5,1,2,3,0,5,0,4,1,2,5,4,1,2,3,4,1,2,0,0,5,4,1,2,5,0,3,3,3,0,5,0,0,0,5,4,3,0,0,3,4,1,2,5,4,1,2,5,4,3,4,3,4,1,2,5,1,2,0,0,3,4,1,2,0,3,4,3,4,1,2,5,4,1,2,3,0,3,3,3,4,1,2,3,3,4,1,2,5,4,1,2,0,3,0,5,4,3,4,1,2,5,3,0,5,0,0,5,0,5,4,1,2,5,3,4,1,2,4,1,2,5,0,3,4,1,2,5,4,3,4,1,2,5,0,0,5,0,3,4,1,2,5,4,1,2,5,4,1,2,5,1,2,0,0,5,4,1,2,5,3,4,3,4,1,2,5,3,4,1,2,5,1,2,5,4,1,2,0,5,4,1,2,5,4,1,2,5,1,2,0,3,0,5,1,2,3,3,4,1,2,5,0,5,3,4,1,2,5,4,1,2,0,5,1,2,0,4,1,2,5,1,2,3,4,1,2,5,4,1,2,5,0,5,1,2,5,0,5,1,2,0,0,5,4,1,2,5,0,0,3,4,3,4,1,2,5,4,1,2,0,5,4,3,4,3,3,4,1,2,4,1,3,0,5,4,1,3,3,3,0,5,4,1,2,5,1,2,0,0,0,0,5,4,1,2,4,1,2,3,4,3,4,3,3,0,0,3,4,1,2,3,3,4,1,2,5,0,3,0,3,3,4,1,2,3,4,1,2,5,4,1,2,3,4,1,2,3,4,1,2,5,0,5,3,3,0,0,3,4,3,0,3,0,0,5,0,5,3,4,1,2,0,3,4,1,2,3]
[7,0,6,4,5,1,2,4,5,1,7,3,0,6,0,6,4,5,1,0,6,4,5,1,3,7,3,2,4,5,1,7,3,3,2,0,6,4,5,1,3,0,0,6,4,5,1,0,6,7,2,0,6,5,1,3,3,3,3,2,4,5,1,3,3,0,6,5,1,3,7,0,6,4,5,1,3,3,3,3,0,6,4,5,1,0,6,7,3,3,3,7,3,0,7,0,0,6,4,7,7,3,2,2,7,6,5,1,3,3,3,7,6,0,6,4,5,1,2,7,0,0,6,7,7,6,5,1,3,3,3,2,4,5,1,7,6,4,5,1,3,0,6,4,5,1,7,0,6,7,0,6,4,5,1,7,6,4,5,1,3,2,7,0,6,4,5,1,0,6,4,5,1,0,0,6,4,5,1,3,3,2,7,7,6,4,5,1,2,4,5,1,0,6,4,5,1,0,6,4,5,1,3,3,0,0,6,4,5,1,3,2,0,7,2,4,5,1,3,3,2,0,6,4,5,1,0,7,6,4,5,1,3,3,0,6,4,5,1,0,6,4,5,1,3,2,4,5,1,7,3,2,0,6,4,5,1,0,6,4,7,7,3,0,6,5,1,2,6,4,5,1,7,0,6,4,5,1,3,3,3,2,4,5,1,3,7,3,3,2,3,2,4,5,1,3,0,0,6,4,5,1,7,2,2,6,4,5,1,2,6,4,5,1,2,7,0,6,0,0,6,4,5,1,3,3,3,3,1,2,7,0,6,4,5,1,0,2,4,5,1,0,6,0,6,4,7,2,4,7,6,4,5,1,3,7,3,0,0,0,0,6,0,0,7,2,4,5,1,2,2,0,0,6,4,5,1,3,0,0,0,6,4,5,1,2,0,0,0,6,4,5,1,2,6,7,3,3,2,4,5,1,3,3,7,7,6,7,7,6,0,0,0,7,0,6,4,5,1,2,0,0,6,4,5,1,2,4,5,1,7,3,3,3,0,6,4,5,1,0,6,4,5,1,3,3,1,0,6,0,6,4,5,1,2,4,5,1,2,6,4,5,1,3,7,6,4,5,1,2,4,5,1,2,7,2,4,5,1,0,6,4,5,1,0,6,4,5,1,3,2,6,7,2,4,5,1,2,0,6,4,5,1,2,4,5,1,2,2,0,0,6,4,5,1,2,2,4,5,1,0,0,6,4,5,1,3,3,3,3,7,6,4,5,1,3,3,2,2,7,2,2,7,2,4,5,1,2,2,2,2,4,5,1,3,0,6,4,5,1]
[2,2,6,3,5,7,0,1,1,6,6,3,5,7,0,4,5,7,0,1,4,3,6,2,3,2,2,4,6,3,5,7,0,2,2,4,6,6,3,5,7,0,1,2,1,6,2,1,4,4,5,7,0,4,3,5,7,0,4,5,7,0,6,3,5,7,0,1,1,4,2,2,1,4,4,1,1,1,6,3,5,7,0,4,6,6,6,3,7,0,4,6,6,3,2,3,5,7,0,1,6,6,4,5,2,1,4,1,1,6,3,2,6,3,5,7,0,1,1,1,1,2,2,2,4,4,2,3,5,7,0,6,3,5,7,0,4,3,6,3,5,7,0,2,1,4,3,2,6,3,5,7,0,4,6,3,2,1,4,4,4,3,7,0,6,2,3,5,7,0,4,4,4,2,3,5,7,0,2,2,6,6,6,3,5,7,0,4,4,4,5,7,0,2,1,1,6,3,5,7,0,1,1,6,6,6,3,7,0,1,6,3,5,7,2,2,2,1,2,6,2,3,5,7,0,1,1,4,5,7,0,6,3,2,2,3,5,7,0,1,4,6,3,5,7,0,1,4,2,2,1,1,6,6,3,5,7,0,6,3,5,7,0,6,3,6,3,5,7,0,4,4,6,6,2,4,5,7,0,4,5,7,0,1,1,6,2,4,6,3,5,7,0,1,6,3,5,7,0,4,5,7,0,6,3,5,7,0,1,6,3,5,7,0,6,2,1,4,4,4,4,5,7,0,1,1,6,6,3,6,3,5,7,0,4,6,2,6,3,5,2,3,5,7,0,1,2,6,3,5,7,0,2,2,6,3,7,0,1,2,4,6,4,5,7,0,4,4,3,5,7,0,4,5,7,0,2,4,3,5,7,0,6,3,5,6,3,5,7,0,2,3,2,6,3,6,3,5,7,0,1,4,5,7,0,6,3,5,7,0,4,5,7,0,4,5,7,0,1,6,3,5,7,0,1,1,4,5,7,0,1,2,2,3,2,2,3,5,7,0,2,4,6,3,5,2,6,6,6,2,4,5,7,0,6,3,5,7,0,2,1,1,2,1,6,3,5,7,0,4,6,3,5,7,0,4,6,6,6,3,2,4,2,3,6,3,5,7,0,6,3,5,7,0,1,4,6,3,7,0,2,3,5,7,0,4,6,2,3,5,7,0,4,2,6,2,3,2,1,4,2,4,5,2,4,4,5,2,6,3,5,7,0,1,1,6,3,5,7,0,6,4,6,3,2,1,6,3,5,7,0,4,5,7,0,4,6,6,3,5]
[1,3,3,1,1,3,1,3,1,1,1,2,0,4,2,3,1,3,1,1,2,0,4,3,3,0,4,2,0,4,2,0,4,1,1,2,3,0,4,1,2,0,4,3,2,0,4,1,2,0,4,1,2,0,4,2,0,4,2,0,4,2,3,0,4,2,0,4,3,3,0,4,3,1,1,1,2,0,4,1,3,0,4,2,3,3,1,2,0,4,3,3,3,1,2,0,4,1,1,2,3,1,1,2,0,4,2,1,1,3,1,3,1,2,0,4,2,0,4,1,2,0,4,2,0,4,2,3,1,2,1,3,0,4,1,1,2,1,3,0,4,2,0,4,3,3,0,4,3,3,1,1,1,1,1,2,0,4,2,0,4,2,0,4,3,1,1,2,0,4,2,0,4,3,3,1,1,3,3,0,4,2,3,1,2,3,3,3,1,2,0,4,2,3,0,4,1,2,3,0,4,2,0,4,2,0,4,3,1,1,3,1,1,2,0,4,1,3,3,1,2,3,1,3,3,1,2,0,4,2,3,1,2,0,4,1,1,1,2,0,4,2,0,4,1,3,1,2,0,4,2,3,1,2,0,4,2,0,4,3,0,4,2,0,4,3,1,2,0,4,2,0,4,2,0,4,3,3,1,2,0,4,2,0,4,2,1,2,0,4,3,1,3,1,2,0,4,2,0,4,3,1,1,2,3,1,3,1,3,3,1,1,1,3,3,3,3,1,2,0,4,2,0,4,2,0,4,3,0,4,3,1,2,0,4,1,1,2,3,1,3,1,3,1,1,3,1,2,0,4]
[3,6,2,1,5,3,4,5,3,2,1,0,3,6,0,0,3,6,2,1,5,3,6,0,4,5,3,2,1,5,3,4,6,2,1,5,3,4,6,2,1,5,6,2,1,5,6,2,1,4,4,5,4,5,0,3,0,3,4,6,2,1,5,0,3,6,2,1,5,0,0,3,4,5,4,5,6,2,1,4,4,0,0,4,4,0,0,3,6,2,1,6,2,1,5,6,2,1,4,5,3,4,5,6,2,1,4,6,2,1,5,4,5,3,6,2,1,0,3,2,1,5,0,0,3,6,2,1,6,2,1,5,0,0,3,6,2,1,5,3,6,2,1,5,0,3,6,2,1,5,3,6,2,1,5,6,2,1,5,0,0,3,6,2,1,5,3,2,1,0,3,6,2,1,4,6,2,1,5,4,5,4,6,2,1,0,3,5,3,2,1,5,4,4,5,3,6,2,1,5,3,6,0,3,4,5,6,2,1,6,2,1,5,4,0,3,2,1,5,3,6,2,1,5,3,0,4,0,3,6,2,1,5,6,2,1,5,6,2,1,5,4,6,2,1,4,4,5,3,6,0,4,5,0,3,6,2,1,5,3,6,2,1,5,3,0,0,0,3,5,4,5,3,4,6,2,1,5,3,0,3,6,2,1,4,6,2,1,4,4,4,5,6,2,1,0,0,4,5,6,2,1,0,3,4,5,4,0,3,6,2,1,5,3,6,2,1,5,0,3,4,5,4,4,6,2,1,4,4,6,2,1,5,0,4,6,2,1,5,3,0,3,6,2,1,0,3,4,0,3,6,2,1,5,3,0,3,4,0,3,6,2,1,5,3,6,2,1,5,4,4,6,2,1,0,3,6,2,1,5,6,2,1,5,0,3,6,2,1,5,3,0,3,6,2,1,5,6,2,1,4,0,3,6,2,1,5,0,0,4,5,6,2,1,0,0,0,0,3,4,4,0,0,3,0,3,4,4,6,2,1,0,3,6,2,1,5,0,0,3,0,3,4,6,2,1,5,4,5,4,6,0,0,0,3,4,5,6,2,1,6,2,1,5,3,0,3,6,2,1,4,5,3,6,2,1,4,4,0,0,3,0,0,0,0,0,3,4,4,5]
[3,2,1,3,0,2,1,3,0,2,1,3,2,2,0,1,3,2,0,1,3,0,1,3,0,1,3,0,1,3,2,1,2,1,3,2,1,3,2,1,3,2,2,1,3,2,0,1,3,2,0,2,0,2,0,2,0,0,1,3,2,0,2,2,1,3,2,2,1,3,0,1,3,2,1,3,0,1,2,2,1,2,0,1,3,2,2,2,2,2,2,0,1,3,2,0,1,3,2,0,1,3,0,1,3,0,1,3,0,1,3,2,2,1,3,0,1,3,0,1,3,0,1,3,0,2,0,1,3,0,1,3,2,0,0,2,1,3,0,1,3,0,1,3,0,1,3,2,0,1,3,2,1,3,3,0,1,3,0,2,1,3,2,2,2,2,0,1,3,0,1,3,2,1,3,0,1,3,2,0,2,2,2,0,2,2,0,1,2,0,1,3,0,1,3,0,1,3,0,2,2,0,1,3,0,2,1,3,2,0,0,1,3,2,1,3,0,1,3,2,1,3,0,1,3,2,0,1,3,0,1,3,0,0,1,3,2,1,3,2,1,3,0,0,1,3,0,0,1,3,2,0,1,3,0,1,3,2,1,3,2,1,3,0,1,3,2,0,2,0,2,2,2,2,0,0,1,3,0,1]
[1,3,4,1,3,2,3,3,4,3,0,3,4,1,2,4,1,2,4,1,3,2,4,1,2,4,1,2,3,0,0,2,4,1,2,3,0,2,4,1,3,0,0,3,0,2,3,4,3,0,2,4,1,2,4,1,3,0,3,0,3,4,1,2,4,1,3,0,0,3,3,4,1,0,0,2,4,1,2,0,2,4,1,0,2,4,1,0,3,0,2,4,1,0,2,4,1,3,0,2,4,1,0,2,4,1,0,2,3,0,3,0,0,2,4,1,2,4,1,0,2,4,1,2,3,3,4,1,3,0,0,3,3,3,0,2,4,1,2,4,1,3,3,0,2,3,4,1,2,4,1,3,0,2,3,0,0,0,2,4,1,0,2,4,1,3,4,1,2,3,0,3,3,3,2,3,3,0,0,2,3,4,1,3,1,3,3,4,1,2,4,1,0,2,4,1,2,4,1,0,2,4,1,2,0,0,3,0,3,2,4,1,0,3,0,0,2,4,1,2,3,0,3,4,1,3,0,0,0,3,3,0,3,3,0,2,4,1,2,4,1,3,0,2,3,2,3,0,2,3,3,0,2,4,1,2,3,4,1,2,4,1,2,3,2,3,0,0,2,4,1,0,2,4,1,2,3,4,1,3,2,4,1,0,2,4,1,3,4,1,2,4,1,0,0,0,2,3,0,3,0,2,4,1,2,4,1,3,0,2,4,1,2,4,1,0,0,3,4,1,2,4,1,0,2,3,3,0,0,2,4,1,2,4,1,2,4,1,3,0,3,0,0,2,4,1,2,4,1,3]
[2,3,0,5,2,3,4,0,6,5,2,3,0,1,1,4,5,2,3,1,4,0,6,0,1,1,1,6,4,0,4,1,6,5,2,3,4,5,2,3,1,6,5,2,3,4,5,1,6,2,3,0,5,2,3,0,6,5,2,3,0,1,6,1,6,1,6,2,3,1,6,0,5,2,3,0,1,4,0,6,5,2,3,0,5,2,3,0,5,1,6,0,5,2,3,0,5,2,3,0,4,1,6,1,6,4,1,6,3,0,4,0,1,6,1,1,6,2,3,5,1,6,0,6,5,2,3,4,1,6,5,2,3,1,1,6,4,5,2,3,4,5,2,3,0,6,1,6,0,1,4,0,6,1,6,1,6,2,3,4,1,6,4,0,6,4,0,4,5,2,3,0,5,2,3,0,5,2,3,1,6,0,4,5,2,3,0,6,4,0,5,2,3,1,4,0,4,4,4,4,4,4,4,0,1,1,6,5,1,1,6,5,2,3,0,5,2,3,1,6,5,2,3,0,5,2,3,0,5,1,6,4,4,0,5,1,6,5,2,3,0,6,1,6,0,1,6,5,2,3,1,4,5,2,3,1,1,1,6,1,1,6,1,6,0,4,0,1,4,1,4,0,6,1,1,6,4,0,6,4,0,5,2,3,1,6,1,4,5,2,3,4,4,0,6,5,2,3,1,1,6,2,3,0,5,1,1,6,2,3,4,0,5,2,3,5,2,3,1,1,6,5,1,6,1,6,5,2,3,1,6,0,1,6,5,1,6,5,2,3,4,4,4,4,0,4,0,4,0,6,4,1,6,3,1,6,0,5,2,3,0,5,2,3,0,5,1,1,6,3,5,2,3,5,2,3,0,5,2,3,0,4,0,5,2,3,0,4,1,1,6,2,3,1,1,1,6,2,3,0,6,1,6,2,3,5,2,3,0,6,5,2,3,0,1,1,4,5,2,3,1,6,1,6,3,0,4,0,4,4,0,4,0,4,4,4,0,1,1,6,5,2,3,0,6,2,3,5,2,3,0,5,2,3,0,5,2,3,0,4,1,4,0,4,0,5,2,3,4,0,5,2,3,4,4,5,2,3,0,5,2,3,0,6,3,0,5,2,3,0]